In [1]:
import cv2
import numpy as np

In [4]:
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")


ball = cv2.imread("ball.png", cv2.IMREAD_UNCHANGED)
nose = cv2.imread("nose.png", cv2.IMREAD_UNCHANGED)


if ball is None or nose is None:
    print("❌ ball.png یا nose.png did not find")
    exit()


if ball.shape[2] == 3:
    ball = cv2.cvtColor(ball, cv2.COLOR_BGR2BGRA)

if nose.shape[2] == 3:
    nose = cv2.cvtColor(nose, cv2.COLOR_BGR2BGRA)

cap = cv2.VideoCapture(0)


def overlay(img, overlay_img, x, y, w, h):

    if w <= 0 or h <= 0:
        return img

    overlay_img = cv2.resize(overlay_img, (w, h))

    x_end = min(x + w, img.shape[1])
    y_end = min(y + h, img.shape[0])

    for i in range(y_end - y):
        for j in range(x_end - x):

            if overlay_img[i, j][3] > 0:
                img[y+i, x+j] = overlay_img[i, j][:3]

    return img


while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(80, 80)
    )

    for (x, y, w, h) in faces:

        # left eye
        size_eye = int(w * 0.22)

        bx1 = x + int(w * 0.22)
        by1 = y + int(h * 0.3)

        # right eye
        bx2 = x + int(w * 0.62)
        by2 = y + int(h * 0.3)

        frame = overlay(frame, ball, bx1, by1, size_eye, size_eye)
        frame = overlay(frame, ball, bx2, by2, size_eye, size_eye)

        
        nw = int(w * 0.28)
        nh = int(h * 0.22)

        nx = x + int(w * 0.36)
        ny = y + int(h * 0.5)

        frame = overlay(frame, nose, nx, ny, nw, nh)

        
        cv2.rectangle(frame, (x,y), (x+w,y+h), (255,0,0), 2)

    cv2.imshow("Ball Face Filter", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()